# Carry, roll-down and returns

Research question: Can every dollar of a fixed holding be reconciled?

Inputs: frozen Treasury 2024–2025 CSVs. Rates are decimals, prices per 100 face. Release 1.0.0. No live signal.

In [1]:
import sys,json
from pathlib import Path
from datetime import date
import numpy as np
R=Path.cwd()
if R.name=='notebooks':R=R.parent
sys.path.insert(0,str(R/'src'));sys.path.insert(0,str(R/'scripts'))
from curve_lab.data import load_history
from curve_lab.bonds import Bond
from curve_lab.curves import bootstrap,par_rate,TENORS,KEYS,fit_nelson_siegel
from curve_lab.risk import bond_risk,hedge,carry_bridge
from curve_lab.forecast import predict,evaluate
from review_display import table,figure
if 'display_bundle' not in globals():
    from IPython.display import display
    def display_bundle(bundle):display(bundle,raw=True)
D=json.loads((R/'artifacts/release/research.json').read_text());rows,receipts=load_history(R)
print('Run',D['run_id'],'| source through',D['as_of'],'| hypothetical bond valuations')


Run p4-662f58a64e8e | source through 2025-12-31 | hypothetical bond valuations


## Rebuild the fixed holding bridge

In [2]:
start=date(2025,1,2);end=date(2025,7,2);c0=bootstrap(next(r['par'] for r in rows if r['date']==str(start)));c1=bootstrap(next(r['par'] for r in rows if r['date']==str(end)));b=Bond(date(2030,1,2),.04);r=carry_bridge(b,start,end,c0,c1,.04,2);display_bundle(table(['Component','Per 100 face'],[[k,r[k]] for k in ['carry','roll','curve_move','financing','transaction_cost','net_pnl','reconciliation']]));assert abs(r['reconciliation'])<1e-10

Component,Per 100 face
carry,2.154094
roll,0.071257484
curve_move,2.0862785
financing,-1.9497989
transaction_cost,-0.039781476
net_pnl,2.3220495
reconciliation,0


## Check the cash identity separately

In [3]:
ledger=r['coupons']+r['end_dirty']-r['start_dirty']+r['financing']+r['transaction_cost'];print('Coupon cash',r['coupons'],'Ledger net',ledger);assert r['coupons']==2;assert abs(ledger-r['net_pnl'])<1e-10

Coupon cash 2.0 Ledger net 2.3220494853775753


## Challenge financing and round-trip costs

In [4]:
display_bundle(table(['Financing','Cost bp per side','Net P&L'],[[f,cost,carry_bridge(b,start,end,c0,c1,f,cost)['net_pnl']] for f,cost in [(0,0),(.04,2),(.08,10)]]))

Financing,Cost bp per side,Net P&L
0,0,4.3116299
0.04,2,2.3220495
0.08,10,0.21312465


## Interpretation

Coupons and accrued interest are not double counted. The ordered bridge is descriptive, and its labels depend on the chosen sequence. Constant-maturity rebalancing requires a separate trade and cost ledger.

Canonical evidence: `artifacts/release/research.json`; source receipts: `data/manifests/sources.json`.